### Import Dependencies

In [ ]:
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
import os

from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.adk.agents.run_config import RunConfig

from google.genai import types

from dotenv import load_dotenv

load_dotenv("../../.env")

from utils.tools_2 import check_warehouse_availability, reserve_warehouse_items

### ADK Agent

In [ ]:
model = LiteLlm(
    model="openai/gpt-5.4-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
)

In [ ]:
warehouse_agent = Agent(
    name="warehouse_manager_agent",
    model=model,
    tools=[
        check_warehouse_availability,
        reserve_warehouse_items
    ],
    description="The user is asking to reserve items from the warehouses or about availability of the items in warehouses.",
    instruction="""You are a part of the shopping assistant that can manage available inventory in the warehouses.

## Instructions

- As the final answer you should return an answer to the users query in a form of actions performed.
- You must always check the availability of the items in the warehouses before reserving them.
- Only reserve items in warehouses if entire order can be reserved or the user has confirmed that they want a partial reservation.
- If you cannot reserve any items, return an answer that the order cannot be reserved.
- If you can reserve some items, return an answer that the order can be partially reserved and include the details.
- If only partial quantity can be reserved in some warehouses, try to combine the required quantity from different warehouses.
- Try to reserve items from the closest warehouse to the user first if users location is provided.
- As the final answer you should return an answer in a form of actions performed.
"""
)


### ADK Session

In [ ]:
session_service = InMemorySessionService()

In [ ]:
await session_service.create_session(
    app_name="warehouse_app",
    user_id="user_1",
    session_id="sessions_1"
)

In [ ]:
runner = Runner(
    agent=warehouse_agent,
    session_service=session_service,
    app_name="warehouse_app"
)

In [ ]:
message = types.Content(
    role="user",
    parts=[
        types.Part(
            text="What is the availability of B09X1LDMH6 in all of your warehouses?"
        )
    ]
)

In [ ]:
result = runner.run(
    user_id="user_1",
    session_id="sessions_1",
    new_message=message,
    run_config=RunConfig(
        max_llm_calls=5
    )
)  

In [ ]:
result

In [ ]:
for event in result:
    print("======================")
    print(event)

### Agent Run Wrapper

In [ ]:
async def run_warehouse_agent(query: str, session_id: str, session_service: InMemorySessionService) -> str:

    existing_session = await session_service.get_session(
        app_name="warehouse_app",
        user_id="user_1",
        session_id=session_id
    )

    if not existing_session:
        await session_service.create_session(
            app_name="warehouse_app",
            user_id="user_1",
            session_id=session_id
        )

    runner = Runner(
        agent=warehouse_agent,
        session_service=session_service,
        app_name="warehouse_app"
    )

    content = types.Content(
        role="user",
        parts=[
            types.Part(
                text=query
            )
        ]
    )

    final_text = ""
    for event in runner.run(
        user_id="user_1",
        session_id=session_id,
        new_message=content,
        run_config=RunConfig(
            max_llm_calls=5
        )
    ):
        if event.is_final_response():
            if event.content and event.content.parts:
                for part in event.content.parts:
                    final_text += part.text

    return final_text

In [ ]:
answer_1 = await run_warehouse_agent(
    query="What is the availability of B09X1LDMH6 in all of your warehouses?",
    session_id="test_01",
    session_service=session_service
)

In [ ]:
print(answer_1)

In [ ]:
answer_2 = await run_warehouse_agent(
    query="Can you reserve 9 of this in the Munich warehouse?",
    session_id="test_01",
    session_service=session_service
)

In [ ]:
print(answer_2)